In [33]:
import os
import re
import numpy as np
import pandas as pd
import torch
from typing import List, Dict, Union, Optional
from dataclasses import dataclass

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import evaluate
from sklearn.model_selection import train_test_split


In [34]:
class SupportTicketDataLoader:
    """Handles loading and preprocessing with dynamic label detection"""
    
    def __init__(self, dataset_name: str = "Tobi-Bueck/customer-support-tickets"):
        self.dataset_name = dataset_name
        self.label2id = {}
        self.id2label = {}
        self.num_labels = 0
        
    def load_and_preprocess(self, test_size: float = 0.15, 
                           val_size: float = 0.15, 
                           random_state: int = 42) -> Dict[str, Dataset]:
        """
        Load dataset, detect labels dynamically, and create splits
        """
        raw_dataset = load_dataset(self.dataset_name, split="train")
        df = raw_dataset.to_pandas()
        
        df = df[['subject', 'body', 'queue']].copy()
        df['text'] = (df['subject'].fillna('') + ' ' + df['body'].fillna('')).str.strip()
        df = df[df['text'].str.len() > 10].copy()  # Remove very short texts
        
        unique_queues = sorted(df['queue'].unique())
        self.label2id = {label: idx for idx, label in enumerate(unique_queues)}
        self.id2label = {idx: label for label, idx in self.label2id.items()}
        self.num_labels = len(unique_queues)  # This will be 52 for this dataset
        
        print(f"Detected {self.num_labels} unique queue labels:")
        for idx in range(min(10, self.num_labels)):  # Show first 10
            print(f"   {idx}: {self.id2label[idx]}")
        if self.num_labels > 10:
            print(f"   ... and {self.num_labels - 10} more")
        
        df['label'] = df['queue'].map(self.label2id)
        
        train_val_df, test_df = train_test_split(
            df, test_size=test_size, random_state=random_state, 
            stratify=df['queue']
        )
        train_df, val_df = train_test_split(
            train_val_df, test_size=val_size/(1-test_size), random_state=random_state,
            stratify=train_val_df['queue']
        )
        
        print(f"\nDataset splits:")
        print(f"Train: {len(train_df):,} samples")
        print(f"Validation: {len(val_df):,} samples")  
        print(f"Test: {len(test_df):,} samples")
        
        def to_hf_dataset(df_subset):
            return Dataset.from_pandas(df_subset[['text', 'label']].reset_index(drop=True))
        
        return {
            'train': to_hf_dataset(train_df),
            'validation': to_hf_dataset(val_df),
            'test': to_hf_dataset(test_df)
        }
    
    def get_label_info(self) -> tuple:
        """Return label mappings and count"""
        return self.label2id, self.id2label, self.num_labels

In [35]:
@dataclass
class TrainingConfig:
    """Configuration - num_labels set dynamically from data"""
    model_name: str = "distilbert-base-uncased"
    # num_labels: int = 10
    learning_rate: float = 2e-5
    batch_size: int = 16
    num_epochs: int = 3
    max_length: int = 512
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    output_dir: str = "./support-ticket-model"
    evaluation_strategy: str = "epoch"
    save_strategy: str = "epoch"
    load_best_model_at_end: bool = True
    metric_for_best_model: str = "f1"
    early_stopping_patience: int = 2

In [36]:
class TicketTokenizer:    
    def __init__(self, model_name: str = "distilbert-base-uncased", max_length: int = 512):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_length = max_length
        
    def tokenize_function(self, examples: Dict) -> Dict:
        return self.tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_special_tokens_mask=True
        )
    
    def encode_single(self, text: str) -> Dict:
        return self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

In [40]:
class SupportTicketClassifier:
    """Main classifier with dynamic label support"""
    
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_name)
        self.model = None
        self.trainer = None
        self.label2id = {}
        self.id2label = {}
        self.num_labels = 0
        
    def setup_model(self, label2id: Dict[str, int], id2label: Dict[int, str], num_labels: int):
        """Initialize model with dynamically detected labels"""
        self.label2id = label2id
        self.id2label = id2label
        self.num_labels = num_labels
        
        print(f"Initializing model with {num_labels} output classes...")
        
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.config.model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
            problem_type="single_label_classification",
            ignore_mismatched_sizes=True  # Handle pretrained head mismatch
        )
        
    def tokenize_function(self, examples):
        """Tokenize texts for model input"""
        return self.tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=self.config.max_length,
        )
    
    def prepare_datasets(self, datasets: Dict[str, Dataset]) -> Dict[str, Dataset]:
        """Tokenize all splits"""
        tokenized = {}
        for split, dataset in datasets.items():
            tokenized[split] = dataset.map(
                self.tokenize_function,
                batched=True,
                remove_columns=['text']
            )
        return tokenized
    
    def compute_metrics(self, eval_pred):
        """Compute evaluation metrics"""
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        
        metrics = {}
        for metric_name in ["accuracy", "f1", "precision", "recall"]:
            metric = evaluate.load(metric_name)
            result = metric.compute(
                predictions=predictions, 
                references=labels, 
                average="weighted" if metric_name != "accuracy" else None
            )
            metrics[metric_name] = result[metric_name]
        return metrics
    
    def train(self, train_dataset: Dataset, eval_dataset: Dataset):
        """Train the model"""
        if self.model is None:
            raise ValueError("Call setup_model() first with label info")
        
        training_args = TrainingArguments(
            output_dir=self.config.output_dir,
            learning_rate=self.config.learning_rate,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            num_train_epochs=self.config.num_epochs,
            weight_decay=self.config.weight_decay,
            eval_strategy=self.config.evaluation_strategy,  # ← CHANGED
            save_strategy=self.config.save_strategy,
            load_best_model_at_end=self.config.load_best_model_at_end,
            metric_for_best_model=self.config.metric_for_best_model,
            warmup_ratio=self.config.warmup_ratio,
            logging_dir=f"{self.config.output_dir}/logs",
            logging_steps=100,
            report_to="none",
            fp16=torch.cuda.is_available(),
        )
        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        
        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=self.config.early_stopping_patience)]
        )
        
        print("Starting training...")
        result = self.trainer.train()
        
        # Save model
        self.trainer.save_model(self.config.output_dir)
        self.tokenizer.save_pretrained(self.config.output_dir)
        
        # Save label mappings for inference
        import json
        with open(f"{self.config.output_dir}/label_mappings.json", "w") as f:
            json.dump({
                "label2id": self.label2id,
                "id2label": {str(k): v for k, v in self.id2label.items()},
                "num_labels": self.num_labels
            }, f, indent=2)
        
        print(f"Training complete. Model saved to {self.config.output_dir}")
        return result
    
    def predict(self, texts: Union[str, List[str]], return_proba: bool = False):
        """Predict labels for new tickets"""
        if self.model is None:
            raise ValueError("Model not initialized")
        
        if isinstance(texts, str):
            texts = [texts]
            single = True
        else:
            single = False
            
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.config.max_length,
            return_tensors="pt"
        )
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(device)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(logits, dim=-1)
        
        results = []
        for i, pred in enumerate(preds):
            label = self.id2label[pred.item()]
            if return_proba:
                results.append({
                    'label': label,
                    'confidence': round(probs[i][pred].item(), 4),
                    'all_probabilities': {
                        self.id2label[j]: round(probs[i][j].item(), 4)
                        for j in range(self.num_labels)
                    }
                })
            else:
                results.append(label)
        
        return results[0] if single else results
    
    def load(self, model_path: str):
        """Load a trained model with its label mappings"""
        import json
        
        # Load label mappings
        with open(f"{model_path}/label_mappings.json", "r") as f:
            mappings = json.load(f)
        
        self.label2id = mappings["label2id"]
        self.id2label = {int(k): v for k, v in mappings["id2label"].items()}
        self.num_labels = mappings["num_labels"]
        
        # Load model
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            local_files_only=True
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        print(f"Loaded model with {self.num_labels} classes from {model_path}")

In [41]:
def generate_classification_report(classifier: SupportTicketClassifier, test_dataset: Dataset,id2label: Dict) -> str:
    predictions = []
    labels = []
    
    for item in test_dataset:
        pred = classifier.predict(item['text'])
        predictions.append(classifier.label2id[pred])
        labels.append(item['label'])
    
    report = classification_report(
        labels, predictions, 
        target_names=[id2label[i] for i in range(len(id2label))],
        digits=4
    )
    
    cm = confusion_matrix(labels, predictions)
    
    return report, cm

In [42]:
def main():
    """Run the complete corrected pipeline"""
    print("Support Ticket Classifier - Fixed Pipeline\n")
    
    config = TrainingConfig(
        model_name="distilbert-base-uncased",
        learning_rate=2e-5,
        batch_size=16,
        num_epochs=3,
        max_length=512,
        output_dir="./models/support-ticket-classifier"
    )
    
    # Step 1: Load data with dynamic label detection
    print("Loading and preprocessing dataset...")
    data_loader = SupportTicketDataLoader()
    datasets = data_loader.load_and_preprocess()
    label2id, id2label, num_labels = data_loader.get_label_info()
    
    # Step 2: Initialize classifier
    print("Setting up model...")
    classifier = SupportTicketClassifier(config)
    classifier.setup_model(label2id, id2label, num_labels)
    
    # Step 3: Tokenize
    print("Tokenizing datasets...")
    tokenized_datasets = classifier.prepare_datasets(datasets)
    
    # Step 4: Train
    print("Training model...")
    classifier.train(tokenized_datasets['train'], tokenized_datasets['validation'])
    
    # Step 5: Evaluate
    print("Evaluating on test set...")
    results = classifier.trainer.evaluate(tokenized_datasets['test'])
    print(f"Test metrics: {results}")
    
    # Step 6: Detailed report
    print("\nClassification Report (Top 10 classes shown):")
    predictions = []
    labels = []
    for item in datasets['test']:
        pred = classifier.predict(item['text'])
        predictions.append(label2id[pred])
        labels.append(item['label'])
    
    report = classification_report(
        labels, predictions,
        target_names=[id2label[i] for i in range(num_labels)],
        digits=3,
        zero_division=0
    )
    lines = report.split('\n')
    for line in lines[:15]:
        print(line)
    print(f"... ({num_labels - 10} more classes)")
    
    print("\nDemo Predictions:")
    demo_tickets = [
        "Cannot login to account, password reset not working",
        "Refund request pending for 2 weeks, order #12345",
        "Package arrived damaged, need replacement",
        "Payment failed during checkout, card declined",
        "When will my delayed order ship? Tracking not updating"
    ]
    
    for ticket in demo_tickets:
        result = classifier.predict(ticket, return_proba=True)
        print(f"\n'{ticket[:50]}...'")
        print(f"{result['label']} (confidence: {result['confidence']:.2%})")
    
    print(f"\nPipeline completed! Model supports {num_labels} ticket categories.")
    return classifier


if __name__ == "__main__":
    trained_model = main()


Support Ticket Classifier - Fixed Pipeline

Loading and preprocessing dataset...
Detected 52 unique queue labels:
   0: Arts & Entertainment/Movies
   1: Arts & Entertainment/Music
   2: Autos & Vehicles/Maintenance
   3: Autos & Vehicles/Sales
   4: Beauty & Fitness/Cosmetics
   5: Beauty & Fitness/Fitness Training
   6: Billing and Payments
   7: Books & Literature/Fiction
   8: Books & Literature/Non-Fiction
   9: Business & Industrial/Manufacturing
   ... and 42 more

Dataset splits:
Train: 43,234 samples
Validation: 9,265 samples
Test: 9,265 samples
Setting up model...
Initializing model with 52 output classes...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing datasets...


Map: 100%|████████████████████████████████████████████████████████████████| 9265/9265 [00:03<00:00, 2445.14 examples/s]
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_35596\1619280817.py:90: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Training model...
🚀 Starting training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 